## Recon contract for `outputs/250_recon` (my local folder sturcture)

### 0) Purpose

`251_recon_shared_data` is the **single canonical output location** for *derived* electrode-localization artifacts used by plotting, atlas mapping, and group-average visualizations. It is **regenerable** and should be treated as pipeline output, not raw data.

---
## Part 1: Generate and save the trsRAS contact coordonates in my drive using LEPTOVOX. 

In [4]:
EL_COLOR_NAMES = ["blue","deepskyblue","cyan","teal","green","lime","powderblue","forestgreen","lightcyan","darkolivegreen"]
PAT_COLOR_NAMES = ["blueviolet","fuchsia","deeppink","crimson","pink","red","chocolate","gold","purple","saddlebrown","lemonchiffon","lavenderblush","lime","powderblue","forestgreen","lightcyan"]
MICRO_COLOR_NAMES = ["navy","darkslategray","black","darkred","darkolivegreen"]

# In lf_recon_shared_config.py or a small export script:
import json
from matplotlib.colors import to_hex
import matplotlib.colors as mcolors

colors_config = {
    "EL":    [to_hex(mcolors.CSS4_COLORS[c]) for c in EL_COLOR_NAMES],
    "PAT":   [to_hex(mcolors.CSS4_COLORS[c]) for c in PAT_COLOR_NAMES],
    "MICRO": [to_hex(mcolors.CSS4_COLORS[c]) for c in MICRO_COLOR_NAMES],
    "conditions": {
        "audio":   to_hex(mcolors.CSS4_COLORS["crimson"]),
        "picture": to_hex(mcolors.CSS4_COLORS["steelblue"]),
        "reading": to_hex(mcolors.CSS4_COLORS["forestgreen"]),
    }
}

with open("outputs/colors_config.json", "w") as f:
    json.dump(colors_config, f, indent=2)

In [5]:
# ============================================================
# ONE CELL: Fixed LEPTOVOX convention
#   perm  = (0,1,2)  (XYZ as-is)
#   flips = (0,0,1)  (flip k only: k' = (dim_z-1)-k)
#
# Batch over all PAT_* in SHARED_ROOT and write:
#   OUT_ROOT/PAT_XXXX/glassbrain/coords/PAT_XXXX_contacts_tkrRAS.csv
#   OUT_ROOT/PAT_XXXX/glassbrain/png/PAT_XXXX_mosaic_LEPTOVOX.png
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import nibabel as nib
import pyvista as pv
import imageio.v3 as iio
from nibabel.freesurfer.io import read_geometry
from scipy.spatial import cKDTree


# -------------------------
# CONFIG
# -------------------------
SHARED_ROOT = Path(r"\\nasac-m2.unige.ch\m-HumanNeuronLab\#SHARE\To_send_collaborators")  # change to where your PAT_XXX folders are
SHARED_ROOT = Path(r"\\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\SEEG_EXPERIMENTS_BERN\Reconstruction")  # change to where your PAT_XXX folders are

OUT_ROOT    = Path(r"\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\02_FBM_Clustering\outputs\250_recon") # Change to where you want the data to be saved, in drive format

# Fixed convention you determined:
PERM  = (0, 1, 2)            # use columns as-is
FLIPS = (False, False, True) # flip k only

# render params
VIEWS = ("left", "frontal", "right")
WINDOW_SIZE = (1200, 1000)
TRANSPARENT_BG = True
BRAIN_COLOR = "#ead6db"
BRAIN_OPACITY = 0.25
POINT_COLOR = "purple"
POINT_SIZE = 10
POINT_OPACITY = 0.9

OVERWRITE_CSV = True
OVERWRITE_PNG = True

# Helpers: 
#---------
def read_electrode_lines_drop2(path: Path) -> list[str]:
    lines = [ln.strip() for ln in path.read_text(encoding="utf-8", errors="ignore").splitlines() if ln.strip()]
    if len(lines) < 3:
        raise ValueError(f"electrodeNames too short: {path}")
    return lines[2:]  # drop timestamp + header

def parse_name_and_hemi(lines: list[str]):
    """
    lines look like: 'FPG1 D L'
    returns:
      name_raw (full line), name (first token), hemi_expected (L/R/None)
    """
    names_raw, names_clean, hemi = [], [], []
    for ln in lines:
        parts = ln.split()
        nm = parts[0] if len(parts) else ln
        h = None
        if len(parts):
            last = parts[-1].upper()
            if last in ("L", "R"):
                h = last
        names_raw.append(ln)
        names_clean.append(nm)
        hemi.append(h)
    return np.array(names_raw, dtype=object), np.array(names_clean, dtype=object), np.array(hemi, dtype=object)

def read_leptovox_xyz(path: Path) -> np.ndarray:
    rows = []
    for ln in path.read_text(encoding="utf-8", errors="ignore").splitlines():
        t = ln.strip()
        if not t or t.startswith("#"):
            continue
        parts = t.replace(",", " ").split()
        if len(parts) < 3:
            continue
        try:
            rows.append((float(parts[0]), float(parts[1]), float(parts[2])))
        except ValueError:
            continue
    if not rows:
        raise ValueError(f"No numeric rows found in {path}")
    return np.asarray(rows, dtype=float)

def pick_mgz(subj_dir: Path) -> Path:
    for cand in ["brainmask.mgz", "T1.mgz", "orig.mgz"]:
        p = subj_dir / "mri" / cand
        if p.is_file():
            return p
    raise FileNotFoundError(f"Missing brainmask/T1/orig in {subj_dir/'mri'}")



# Geometry helpers
# ----------------
def voxel_to_tkr(points_ijk: np.ndarray, vox2ras_tkr: np.ndarray) -> np.ndarray:
    n = points_ijk.shape[0]
    ijk_h = np.c_[points_ijk, np.ones(n)]
    tkr_h = (vox2ras_tkr @ ijk_h.T).T
    return tkr_h[:, :3]

def apply_voxel_flips(ijk: np.ndarray, vol_shape, flips=(False, False, True)) -> np.ndarray:
    dims = np.array(vol_shape, dtype=float)
    out = ijk.copy()
    for ax, do_flip in enumerate(flips):
        if do_flip:
            out[:, ax] = (dims[ax] - 1.0) - out[:, ax]
    return out

def nearest_pial_metrics(points_tkr: np.ndarray, lh_v: np.ndarray, rh_v: np.ndarray):
    kdl = cKDTree(lh_v)
    kdr = cKDTree(rh_v)
    dl, _ = kdl.query(points_tkr, k=1, workers=-1)
    dr, _ = kdr.query(points_tkr, k=1, workers=-1)
    is_left = dl <= dr
    dist = np.minimum(dl, dr)
    return dist, is_left



# Rendering helpers
# -------------------------
def make_mesh(v, f):
    return pv.PolyData(v, np.c_[np.full(len(f), 3), f].astype(np.int64))

def set_view(pl, view):
    view = view.lower()
    if view == "left":
        pl.view_yz(negative=True)
    elif view == "frontal":
        pl.view_xz(negative=False)
    elif view == "right":
        pl.view_yz(negative=False)
    else:
        raise ValueError(view)
    pl.camera.zoom(1.15)

def render_view(lh_mesh, rh_mesh, points_tkr, view):
    pl = pv.Plotter(off_screen=True, window_size=WINDOW_SIZE)
    pl.set_background("white")
    pl.add_mesh(lh_mesh, color=BRAIN_COLOR, opacity=BRAIN_OPACITY, smooth_shading=True)
    pl.add_mesh(rh_mesh, color=BRAIN_COLOR, opacity=BRAIN_OPACITY, smooth_shading=True)
    pl.add_points(points_tkr, color=POINT_COLOR, render_points_as_spheres=True,
                  point_size=POINT_SIZE, opacity=POINT_OPACITY)
    set_view(pl, view)
    img = pl.screenshot(transparent_background=TRANSPARENT_BG, return_img=True)
    pl.close()
    return img

def stitch_horiz(imgs):
    H = min(im.shape[0] for im in imgs)
    imgs = [im[:H] for im in imgs]
    return np.concatenate(imgs, axis=1)


# Per patient
# -------------------------
def export_and_mosaic_patient(pid: str):
    pid = str(pid)
    subj_dir = SHARED_ROOT / pid

    names_path = subj_dir / "elec_recon" / f"{pid}.electrodeNames"
    vox_path   = subj_dir / "elec_recon" / f"{pid}.LEPTOVOX"
    if not names_path.is_file():
        raise FileNotFoundError(f"{pid}: missing {names_path}")
    if not vox_path.is_file():
        raise FileNotFoundError(f"{pid}: missing {vox_path}")

    mgz = pick_mgz(subj_dir)
    img = nib.load(str(mgz))
    vox2ras_tkr = img.header.get_vox2ras_tkr()
    vol_shape = img.shape[:3]

    # surfaces
    lh_v, lh_f = read_geometry(str(subj_dir / "surf" / "lh.pial"))
    rh_v, rh_f = read_geometry(str(subj_dir / "surf" / "rh.pial"))
    lh_mesh = make_mesh(lh_v, lh_f)
    rh_mesh = make_mesh(rh_v, rh_f)

    # names
    lines = read_electrode_lines_drop2(names_path)
    name_raw, name_clean, hemi_expected = parse_name_and_hemi(lines)

    # leptovox coords
    pts = read_leptovox_xyz(vox_path)
    if pts.shape[0] != len(name_raw):
        raise RuntimeError(f"{pid}: LEPTOVOX rows ({pts.shape[0]}) != electrodeNames contacts ({len(name_raw)})")

    # 0/1-based detection
    mins = pts.min(axis=0)
    maxs = pts.max(axis=0)
    dims = np.array(vol_shape, dtype=float)

    looks_one_based  = (mins >= 1).all() and (maxs <= dims).all()
    looks_zero_based = (mins >= 0).all() and (maxs <  dims).all()

    pts_ijk = pts.copy()
    index_mode = "0-based (assumed)"
    if looks_one_based and not looks_zero_based:
        pts_ijk -= 1.0
        index_mode = "1-based->0-based"

    # apply fixed perm + flips
    pts_ijk = pts_ijk[:, PERM]  # (0,1,2) = no-op, kept for explicitness
    pts_ijk = apply_voxel_flips(pts_ijk, vol_shape, FLIPS)  # flip k only

    # voxel -> tkrRAS
    pts_tkr = voxel_to_tkr(pts_ijk, vox2ras_tkr)

    # dist + hemi prediction
    dist_mm, pred_is_left = nearest_pial_metrics(pts_tkr, lh_v, rh_v)

    # outputs
    out_coords = OUT_ROOT / pid / "glassbrain" / "coords"
    out_pngdir = OUT_ROOT / pid / "glassbrain" / "png"
    out_coords.mkdir(parents=True, exist_ok=True)
    out_pngdir.mkdir(parents=True, exist_ok=True)

    out_csv = out_coords / f"{pid}_contacts_tkrRAS.csv"
    out_png = out_pngdir / f"{pid}_mosaic_LEPTOVOX.png"

    # CSV
    if OVERWRITE_CSV or (not out_csv.is_file()):
        df_out = pd.DataFrame({
            "name_raw": name_raw,
            "name": name_clean,
            "hemi_expected": hemi_expected,
            "x": pts_tkr[:,0], "y": pts_tkr[:,1], "z": pts_tkr[:,2],
            "pred_isLeft": pred_is_left.astype(int),
            "dist_to_pial_mm": np.round(dist_mm, 2),
            "source_space": "tkrRAS",
            "source_provenance": f"LEPTOVOX; {index_mode}; perm={PERM}; flips={tuple(int(b) for b in FLIPS)} (flip k only)",
            "leptovox_file": str(vox_path),
            "mgz_used": str(mgz),
        })
        df_out.to_csv(out_csv, index=False)

    # Mosaic
    if OVERWRITE_PNG or (not out_png.is_file()):
        imgs = [render_view(lh_mesh, rh_mesh, pts_tkr, v) for v in VIEWS]
        mosaic = stitch_horiz(imgs)
        iio.imwrite(out_png, mosaic)

    # quick stats
    med_dist = float(np.median(dist_mm))
    pct_left = float(np.mean(pred_is_left) * 100.0)

    return {
        "pid": pid,
        "status": "OK",
        "n_contacts": int(len(name_raw)),
        "median_dist_to_pial_mm": med_dist,
        "pct_pred_left": pct_left,
        "index_mode": index_mode,
        "csv": str(out_csv),
        "png": str(out_png),
    }


# -------------------------
# Batch run all PAT_* (Careful, this runs ALL patients in the shared folder. make another patient_ids variable with list of patient names ["PAT_3066","" ...]
# -------------------------
patient_ids = sorted([p.name for p in SHARED_ROOT.iterdir() if p.is_dir() and p.name.startswith("EL")])
rows = []
for pid in patient_ids:
    try:
        rows.append(export_and_mosaic_patient(pid))
    except Exception as e:
        rows.append({"pid": pid, "status": "ERROR", "error": str(e)})

df_status = pd.DataFrame(rows)
rows


[{'pid': 'EL030',
  'status': 'OK',
  'n_contacts': 102,
  'median_dist_to_pial_mm': 2.1893364350578577,
  'pct_pred_left': 26.47058823529412,
  'index_mode': '0-based (assumed)',
  'csv': '\\\\nasac-m2.unige.ch\\m-HumanNeuronLab\\ANALYSIS\\FLM\\Analysis_LoraFanda\\02_FBM_Clustering\\outputs\\250_recon\\EL030\\glassbrain\\coords\\EL030_contacts_tkrRAS.csv',
  'png': '\\\\nasac-m2.unige.ch\\m-HumanNeuronLab\\ANALYSIS\\FLM\\Analysis_LoraFanda\\02_FBM_Clustering\\outputs\\250_recon\\EL030\\glassbrain\\png\\EL030_mosaic_LEPTOVOX.png'},
 {'pid': 'EL033',
  'status': 'OK',
  'n_contacts': 92,
  'median_dist_to_pial_mm': 2.934116525023538,
  'pct_pred_left': 26.08695652173913,
  'index_mode': '0-based (assumed)',
  'csv': '\\\\nasac-m2.unige.ch\\m-HumanNeuronLab\\ANALYSIS\\FLM\\Analysis_LoraFanda\\02_FBM_Clustering\\outputs\\250_recon\\EL033\\glassbrain\\coords\\EL033_contacts_tkrRAS.csv',
  'png': '\\\\nasac-m2.unige.ch\\m-HumanNeuronLab\\ANALYSIS\\FLM\\Analysis_LoraFanda\\02_FBM_Clustering\

In [6]:
 SHARED_ROOT.iterdir()

<generator object Path.iterdir at 0x0000018D7DC32B20>

## 250_recon/fsaverage folder outputs: Across all patients

In [7]:
# ============================================================
# Surface-based group mapping: patient tkrRAS → fsaverage
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import pyvista as pv
import imageio.v3 as iio
from nibabel.freesurfer.io import read_geometry
from scipy.spatial import cKDTree

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------
SHARED_ROOT = Path(r"\\nasac-m2.unige.ch\m-HumanNeuronLab\#SHARE\To_send_collaborators")  # change to where your PAT_XXX folders are
# SHARED_ROOT = Path(r"\\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\SEEG_EXPERIMENTS_BERN\Reconstruction")  # change to where your PAT_XXX folders are
OUT_ROOT = Path(r"\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\02_FBM_Clustering\outputs\250_recon")

FSAVERAGE = SHARED_ROOT / "fsaverage"
# FSAVERAGE =  Path(r"\\nasac-m2.unige.ch\m-HumanNeuronLab\#SHARE\To_send_collaborators\fsaverage")

# ------------------------------------------------------------
# LOAD FSAVERAGE SURFACES
# ------------------------------------------------------------
fs_lh_v, fs_lh_f = read_geometry(str(FSAVERAGE / "surf" / "lh.pial"))
fs_rh_v, fs_rh_f = read_geometry(str(FSAVERAGE / "surf" / "rh.pial"))

fs_lh_sph, _ = read_geometry(str(FSAVERAGE / "surf" / "lh.sphere.reg"))
fs_rh_sph, _ = read_geometry(str(FSAVERAGE / "surf" / "rh.sphere.reg"))

fs_lh_kd = cKDTree(fs_lh_sph)
fs_rh_kd = cKDTree(fs_rh_sph)

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------
def make_mesh(v, f):
    return pv.PolyData(v, np.c_[np.full(len(f), 3), f].astype(np.int64))

def stitch_horiz(imgs):
    H = min(im.shape[0] for im in imgs)
    return np.concatenate([im[:H] for im in imgs], axis=1)

def render_fsaverage(points):
    lh = make_mesh(fs_lh_v, fs_lh_f)
    rh = make_mesh(fs_rh_v, fs_rh_f)

    views = []
    for view in ("left", "frontal", "right"):
        pl = pv.Plotter(off_screen=True, window_size=(1200, 1000))
        pl.set_background("white")
        pl.add_mesh(lh, color="#ead6db", opacity=0.25)
        pl.add_mesh(rh, color="#ead6db", opacity=0.25)
        pl.add_points(points, color="purple", point_size=10, render_points_as_spheres=True)

        if view == "left":
            pl.view_yz(negative=True)
        elif view == "right":
            pl.view_yz(negative=False)
        else:
            pl.view_xz()

        img = pl.screenshot(transparent_background=True, return_img=True)
        pl.close()
        views.append(img)

    return stitch_horiz(views)

# ------------------------------------------------------------
# MAIN LOOP
# ------------------------------------------------------------
all_rows = []

patient_ids = sorted([
    p.name for p in OUT_ROOT.iterdir()
    if p.is_dir() and p.name.startswith("PAT_")
])

for pid in patient_ids:
    print(f"Mapping {pid} → fsaverage")

    subj_dir = SHARED_ROOT / pid
    csv_in = OUT_ROOT / pid / "glassbrain" / "coords" / f"{pid}_contacts_tkrRAS.csv"
    if not csv_in.is_file():
        print(f"  SKIP: missing {csv_in}")
        continue

    # Load patient surfaces + spheres
    lh_v, _ = read_geometry(str(subj_dir / "surf" / "lh.pial"))
    rh_v, _ = read_geometry(str(subj_dir / "surf" / "rh.pial"))
    lh_sph, _ = read_geometry(str(subj_dir / "surf" / "lh.sphere.reg"))
    rh_sph, _ = read_geometry(str(subj_dir / "surf" / "rh.sphere.reg"))

    lh_kd = cKDTree(lh_v)
    rh_kd = cKDTree(rh_v)

    df = pd.read_csv(csv_in)
    pts = df[["x", "y", "z"]].to_numpy(float)

    rows_out = []

    for i, p in enumerate(pts):
        # determine hemisphere by nearest pial
        dl, il = lh_kd.query(p)
        dr, ir = rh_kd.query(p)

        if dl <= dr:
            hemi = "L"
            vtx = il
            sph = lh_sph[vtx]
            _, fs_vtx = fs_lh_kd.query(sph)
            fs_xyz = fs_lh_v[fs_vtx]
        else:
            hemi = "R"
            vtx = ir
            sph = rh_sph[vtx]
            _, fs_vtx = fs_rh_kd.query(sph)
            fs_xyz = fs_rh_v[fs_vtx]

        rows_out.append({
            "patient": pid,
            "name": df.loc[i, "name"],
            "hemi": hemi,
            "x": fs_xyz[0],
            "y": fs_xyz[1],
            "z": fs_xyz[2],
        })

    df_fs = pd.DataFrame(rows_out)

    out_dir = OUT_ROOT / "fsaverage" / "coords"
    out_dir.mkdir(parents=True, exist_ok=True)

    out_csv = out_dir / f"{pid}_contacts_fsaverage.csv"
    df_fs.to_csv(out_csv, index=False)

    all_rows.append(df_fs)

# ------------------------------------------------------------
# CONCATENATE + RENDER GROUP BRAIN
# ------------------------------------------------------------
df_all = pd.concat(all_rows, ignore_index=True)

out_all = OUT_ROOT / "fsaverage" / "coords" / "ALL_PATIENTS_contacts_fsaverage.csv"
df_all.to_csv(out_all, index=False)

print(f"Saved group coordinates → {out_all}")

pts_all = df_all[["x", "y", "z"]].to_numpy(float)

mosaic = render_fsaverage(pts_all)

png_dir = OUT_ROOT / "fsaverage" / "png"
png_dir.mkdir(parents=True, exist_ok=True)

png_out = png_dir / "ALL_PATIENTS_fsaverage_mosaic.png"
iio.imwrite(png_out, mosaic)

print(f"Saved group fsaverage mosaic → {png_out}")


Mapping PAT_2868 → fsaverage
Mapping PAT_3066 → fsaverage
Mapping PAT_3301 → fsaverage
Mapping PAT_3390 → fsaverage
Mapping PAT_3415 → fsaverage
Mapping PAT_3455 → fsaverage
Mapping PAT_3780 → fsaverage
Mapping PAT_3965 → fsaverage
Mapping PAT_3975 → fsaverage
Mapping PAT_5515 → fsaverage
Mapping PAT_5533 → fsaverage
Mapping PAT_6619 → fsaverage
Mapping PAT_6684 → fsaverage
Mapping PAT_6704 → fsaverage
Mapping PAT_6854 → fsaverage
Saved group coordinates → \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\02_FBM_Clustering\outputs\250_recon\fsaverage\coords\ALL_PATIENTS_contacts_fsaverage.csv
Saved group fsaverage mosaic → \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\02_FBM_Clustering\outputs\250_recon\fsaverage\png\ALL_PATIENTS_fsaverage_mosaic.png


## 250_recon/talairach outputs

In [8]:
# ============================================================
# Volumetric group mapping: patient tkrRAS → Talairach space
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import pyvista as pv
import imageio.v3 as iio
import re

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------
SHARED_ROOT = Path(r"\\nasac-m2.unige.ch\m-HumanNeuronLab\#SHARE\To_send_collaborators")
OUT_ROOT = Path(r"\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\02_FBM_Clustering\outputs\250_recon")

PATIENT_COLORS = [
    "blueviolet","fuchsia","deeppink","crimson","pink","red",
    "chocolate","gold","purple","saddlebrown","lemonchiffon",
    "lavenderblush","lime","powderblue","forestgreen","lightcyan",
    "navy","darkslategray","black","darkred","darkolivegreen","aquamarine","aquamarine","aquamarine"
]

def render_volumetric_by_patient(df_all):
    """
    Render Talairach/MNI points with one color per patient
    to visually detect bad transforms.
    """
    pl_views = []

    for view in ("left", "frontal", "right"):
        pl = pv.Plotter(off_screen=True, window_size=(1200, 1000))
        pl.set_background("white")
        if "patient" not in df_all.columns:
            df_all["patient"] = df_all.get("pid", None)
        df_all["patient"] = df_all["leptovox_file"].str.extract(r"(PAT_\d+)", expand=False)

        for i, (pid, dfp) in enumerate(df_all.groupby("patient")):
            color = PATIENT_COLORS[i % len(PATIENT_COLORS)]
            pts = dfp[["x", "y", "z"]].to_numpy(float)

            pl.add_points(
                pts,
                color=color,
                point_size=10,
                render_points_as_spheres=True,
                opacity=0.9,
                label=pid,
            )

        if view == "left":
            pl.view_yz(negative=True)
        elif view == "right":
            pl.view_yz(negative=False)
        else:
            pl.view_xz()

        img = pl.screenshot(transparent_background=True, return_img=True)
        pl.close()
        pl_views.append(img)

    # stitch horizontally
    H = min(im.shape[0] for im in pl_views)
    return np.concatenate([im[:H] for im in pl_views], axis=1)


# ------------------------------------------------------------
# TAL TRANSFORM PARSER
# ------------------------------------------------------------
def load_talairach_xfm(xfm_path: Path) -> np.ndarray:
    """
    Robust FreeSurfer talairach.xfm / MNI linear transform parser.
    Handles trailing semicolons and format variants.
    Returns 4x4 affine (RAS -> MNI/Talairach).
    """
    lines = xfm_path.read_text(encoding="utf-8", errors="ignore").splitlines()

    start = None
    for i, ln in enumerate(lines):
        if ln.strip().startswith("Linear_Transform"):
            start = i + 1
            break

    if start is None:
        raise RuntimeError(f"Could not find Linear_Transform in {xfm_path}")

    rows = []
    for j in range(3):
        raw = lines[start + j].strip()
        parts = [p.rstrip(";") for p in raw.split()]
        if len(parts) != 4:
            raise RuntimeError(f"Invalid transform row: {raw}")
        rows.append([float(p) for p in parts])

    M = np.eye(4)
    M[:3, :4] = np.array(rows, dtype=float)
    return M


def apply_affine(points_xyz: np.ndarray, M: np.ndarray) -> np.ndarray:
    n = points_xyz.shape[0]
    xyz_h = np.c_[points_xyz, np.ones(n)]
    out = (M @ xyz_h.T).T
    return out[:, :3]


# ------------------------------------------------------------
# RENDERING
# ------------------------------------------------------------
def render_volumetric(points, color="purple"):
    views = []
    for view in ("left", "frontal", "right"):
        pl = pv.Plotter(off_screen=True, window_size=(1200, 1000))
        pl.set_background("white")
        pl.add_points(points, color=color, point_size=10, render_points_as_spheres=True)

        if view == "left":
            pl.view_yz(negative=True)
        elif view == "right":
            pl.view_yz(negative=False)
        else:
            pl.view_xz()

        img = pl.screenshot(transparent_background=True, return_img=True)
        pl.close()
        views.append(img)

    H = min(im.shape[0] for im in views)
    return np.concatenate([im[:H] for im in views], axis=1)


# ------------------------------------------------------------
# MAIN LOOP
# ------------------------------------------------------------
all_rows = []

patient_ids = sorted([
    p.name for p in OUT_ROOT.iterdir()
    if p.is_dir() and p.name.startswith("PAT_")
])

Patients_ignored = ["PAT_5515"]
patient_ids = sorted([pid for pid in patient_ids if pid not in Patients_ignored])

for pid in patient_ids:
    print(f"Talairach mapping: {pid}")

    subj_dir = SHARED_ROOT / pid
    csv_in = OUT_ROOT / pid / "glassbrain" / "coords" / f"{pid}_contacts_tkrRAS.csv"
    xfm = subj_dir / "mri" / "transforms" / "talairach.xfm"

    if not csv_in.is_file():
        print(f"  SKIP: missing {csv_in}")
        continue
    if not xfm.is_file():
        print(f"  SKIP: missing {xfm}")
        continue

    # Load data
    df = pd.read_csv(csv_in)
    pts = df[["x", "y", "z"]].to_numpy(float)

    # Load talairach transform
    M = load_talairach_xfm(xfm)

    # Apply transform
    pts_tal = apply_affine(pts, M)

    # Save per-patient CSV
    out_dir = OUT_ROOT / "talairach" / "coords"
    out_dir.mkdir(parents=True, exist_ok=True)

    df_out = df.copy()
    df_out[["x", "y", "z"]] = pts_tal
    df_out["space"] = "Talairach"
    df_out["transform"] = "FreeSurfer talairach.xfm"

    out_csv = out_dir / f"{pid}_contacts_talairach.csv"
    df_out.to_csv(out_csv, index=False)

    all_rows.append(df_out)


# # ------------------------------------------------------------
# # CONCATENATE + RENDER GROUP
# # ------------------------------------------------------------
df_all = pd.concat(all_rows, ignore_index=True)

out_all = OUT_ROOT / "talairach" / "coords" / "ALL_PATIENTS_contacts_talairach.csv"
out_all.parent.mkdir(parents=True, exist_ok=True)
df_all.to_csv(out_all, index=False)

print(f"Saved Talairach group CSV → {out_all}")

# Render combined volumetric view
pts_all = df_all[["x", "y", "z"]].to_numpy(float)
mosaic = render_volumetric(pts_all)

png_dir = OUT_ROOT / "talairach" / "png"
png_dir.mkdir(parents=True, exist_ok=True)

png_out = png_dir / "ALL_PATIENTS_talairach_mosaic.png"
iio.imwrite(png_out, mosaic)

print(f"Saved Talairach mosaic → {png_out}")

mosaic = render_volumetric_by_patient(df_all)

png_dir = OUT_ROOT / "talairach" / "png"
png_dir.mkdir(parents=True, exist_ok=True)

png_out = png_dir / "ALL_PATIENTS_talairach_mosaic_colorcoded.png"
iio.imwrite(png_out, mosaic)

print(f"Saved color-coded Talairach mosaic → {png_out}")



Talairach mapping: PAT_2868
Talairach mapping: PAT_3066
Talairach mapping: PAT_3301
Talairach mapping: PAT_3390
Talairach mapping: PAT_3415
Talairach mapping: PAT_3455
Talairach mapping: PAT_3780
Talairach mapping: PAT_3965
Talairach mapping: PAT_3975
Talairach mapping: PAT_5533
Talairach mapping: PAT_6619
Talairach mapping: PAT_6684
Talairach mapping: PAT_6704
Talairach mapping: PAT_6854
Saved Talairach group CSV → \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\02_FBM_Clustering\outputs\250_recon\talairach\coords\ALL_PATIENTS_contacts_talairach.csv
Saved Talairach mosaic → \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\02_FBM_Clustering\outputs\250_recon\talairach\png\ALL_PATIENTS_talairach_mosaic.png
Saved color-coded Talairach mosaic → \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\02_FBM_Clustering\outputs\250_recon\talairach\png\ALL_PATIENTS_talairach_mosaic_colorcoded.png


In [9]:
from pathlib import Path
import os

FSAVERAGE_DIR = Path(r"\\nasac-m2.unige.ch\m-HumanNeuronLab\#SHARE\To_send_collaborators\fsaverage")

def tree(p: Path, max_depth=3, max_items_per_dir=200):
    p = Path(p)
    print(f"\n=== FSAVERAGE TREE ===")
    print(f"root: {p}")
    if not p.exists():
        print("ERROR: path does not exist")
        return

    for root, dirs, files in os.walk(p):
        rel = Path(root).relative_to(p)
        depth = len(rel.parts)
        if depth > max_depth:
            dirs[:] = []
            continue

        indent = "  " * depth
        print(f"{indent}{rel if str(rel) != '.' else '.'}/")

        # show dirs
        dirs_sorted = sorted(dirs)
        if dirs_sorted:
            show_dirs = dirs_sorted[:max_items_per_dir]
            for d in show_dirs:
                print(f"{indent}  [D] {d}")
            if len(dirs_sorted) > max_items_per_dir:
                print(f"{indent}  ... ({len(dirs_sorted)-max_items_per_dir} more dirs)")

        # show files
        files_sorted = sorted(files)
        if files_sorted:
            show_files = files_sorted[:max_items_per_dir]
            for f in show_files:
                # show size to spot template volumes etc.
                fp = Path(root) / f
                try:
                    sz = fp.stat().st_size
                except Exception:
                    sz = None
                if sz is None:
                    print(f"{indent}  [F] {f}")
                else:
                    print(f"{indent}  [F] {f}  ({sz/1e6:.1f} MB)")
            if len(files_sorted) > max_items_per_dir:
                print(f"{indent}  ... ({len(files_sorted)-max_items_per_dir} more files)")

tree(FSAVERAGE_DIR, max_depth=4)



=== FSAVERAGE TREE ===
root: \\nasac-m2.unige.ch\m-HumanNeuronLab\#SHARE\To_send_collaborators\fsaverage
./
  [D] bem
  [D] label
  [D] mri
  [D] mri.2mm
  [D] scripts
  [D] src
  [D] stats
  [D] surf
  [D] tmp
  [D] touch
  [D] trash
  bem/
  label/
    [F] lh.BA1.label  (0.2 MB)
    [F] lh.BA2.label  (0.3 MB)
    [F] lh.BA3a.label  (0.2 MB)
    [F] lh.BA3b.label  (0.2 MB)
    [F] lh.BA44.label  (0.2 MB)
    [F] lh.BA45.label  (0.1 MB)
    [F] lh.BA4a.label  (0.2 MB)
    [F] lh.BA4p.label  (0.2 MB)
    [F] lh.BA6.label  (0.5 MB)
    [F] lh.MT.label  (0.1 MB)
    [F] lh.Medial_wall.label  (0.6 MB)
    [F] lh.PALS_B12.labels.gii  (0.3 MB)
    [F] lh.PALS_B12_Brodmann.annot  (1.3 MB)
    [F] lh.PALS_B12_Lobes.annot  (1.3 MB)
    [F] lh.PALS_B12_OrbitoFrontal.annot  (1.3 MB)
    [F] lh.PALS_B12_Visuotopic.annot  (1.3 MB)
    [F] lh.V1.label  (0.2 MB)
    [F] lh.V2.label  (0.3 MB)
    [F] lh.Yeo2011_17NetworksConfidence_N1000.mgz  (0.5 MB)
    [F] lh.Yeo2011_17Networks_N1000.annot  (1.3 M

## (dont run) EXTRA: Script to loop through different permutations and also flips to see which fits best (i.e. if a flip or permutation is needed)

In [14]:


# # ============================================================
# # ONE CELL: LEPTOVOX -> tkrRAS with (perm + flips) search
# #          hemi-aware scoring + save TOP-K candidate mosaics
# #          so you can visually verify which is correct.
# #
# # Works with:
# #   elec_recon/PAT_XXXX.LEPTOVOX
# #   elec_recon/PAT_XXXX.electrodeNames  (first 2 lines are headers)
# #   mri/brainmask.mgz (or T1/orig)
# #   surf/lh.pial, surf/rh.pial
# #
# # Outputs per patient under OUT_ROOT/PAT_XXXX/glassbrain/:
# #   coords/PAT_XXXX_contacts_tkrRAS.csv
# #   png/PAT_XXXX_mosaic_LEPTOVOX.png                (best)
# #   png_candidates/PAT_XXXX_cand01_...png ...       (top K)
# #   png_candidates/PAT_XXXX_candidates_summary.csv  (table)
# # ============================================================

# from pathlib import Path
# import itertools
# import numpy as np
# import pandas as pd
# import nibabel as nib
# import pyvista as pv
# import imageio.v3 as iio
# from nibabel.freesurfer.io import read_geometry
# from scipy.spatial import cKDTree


# # -------------------------
# # CONFIG
# # -------------------------
# SHARED_ROOT = Path(r"\\nasac-m2.unige.ch\m-HumanNeuronLab\#SHARE\To_send_collaborators")
# OUT_ROOT    = Path(r"\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\02_FBM_Clustering\outputs\250_recon")

# # scoring weights
# LAMBDA_HEMI = 120.0   # mm penalty for hemisphere mismatch (increase if still mirrored)
# K_CANDIDATES_TO_SAVE = 12  # how many candidate mosaics to save per patient

# # render params
# VIEWS = ("left", "frontal", "right")
# WINDOW_SIZE = (1200, 1000)
# TRANSPARENT_BG = True
# BRAIN_COLOR = "#ead6db"
# BRAIN_OPACITY = 0.25
# POINT_COLOR = "purple"
# POINT_SIZE = 10
# POINT_OPACITY = 0.9

# OVERWRITE_CSV = True
# OVERWRITE_PNG = True
# OVERWRITE_CANDIDATES = True


# # -------------------------
# # IO helpers
# # -------------------------
# def read_electrode_lines_drop2(path: Path) -> list[str]:
#     """Return electrode lines after dropping 2 header lines."""
#     lines = [ln.strip() for ln in path.read_text(encoding="utf-8", errors="ignore").splitlines() if ln.strip()]
#     if len(lines) < 3:
#         raise ValueError(f"electrodeNames too short: {path}")
#     return lines[2:]

# def parse_name_and_hemi(lines: list[str]):
#     """
#     lines look like: 'FPG1 D L' or 'FOG12 D R'
#     We keep the whole line as 'name_raw' for traceability and also extract a clean name + hem label.
#     """
#     names_raw = []
#     names_clean = []
#     hemi = []
#     for ln in lines:
#         parts = ln.split()
#         if len(parts) >= 1:
#             nm = parts[0]
#         else:
#             nm = ln
#         h = None
#         # hem is often last token "L" or "R"
#         if len(parts) >= 1:
#             last = parts[-1].upper()
#             if last in ("L", "R"):
#                 h = last
#         names_raw.append(ln)
#         names_clean.append(nm)
#         hemi.append(h)
#     return np.array(names_raw, dtype=object), np.array(names_clean, dtype=object), np.array(hemi, dtype=object)

# def read_leptovox_xyz(path: Path) -> np.ndarray:
#     rows = []
#     for ln in path.read_text(encoding="utf-8", errors="ignore").splitlines():
#         t = ln.strip()
#         if not t or t.startswith("#"):
#             continue
#         parts = t.replace(",", " ").split()
#         if len(parts) < 3:
#             continue
#         try:
#             rows.append((float(parts[0]), float(parts[1]), float(parts[2])))
#         except ValueError:
#             continue
#     if not rows:
#         raise ValueError(f"No numeric rows found in {path}")
#     return np.asarray(rows, dtype=float)

# def pick_mgz(subj_dir: Path) -> Path:
#     for cand in ["brainmask.mgz", "T1.mgz", "orig.mgz"]:
#         p = subj_dir / "mri" / cand
#         if p.is_file():
#             return p
#     raise FileNotFoundError(f"Missing brainmask/T1/orig in {subj_dir/'mri'}")


# # -------------------------
# # Geometry / scoring helpers
# # -------------------------
# def voxel_to_tkr(points_ijk: np.ndarray, vox2ras_tkr: np.ndarray) -> np.ndarray:
#     n = points_ijk.shape[0]
#     ijk_h = np.c_[points_ijk, np.ones(n)]
#     tkr_h = (vox2ras_tkr @ ijk_h.T).T
#     return tkr_h[:, :3]

# def nearest_pial_metrics(points_tkr: np.ndarray, lh_v: np.ndarray, rh_v: np.ndarray):
#     kdl = cKDTree(lh_v)
#     kdr = cKDTree(rh_v)
#     dl, _ = kdl.query(points_tkr, k=1, workers=-1)
#     dr, _ = kdr.query(points_tkr, k=1, workers=-1)
#     is_left = dl <= dr
#     dist = np.minimum(dl, dr)
#     return dist, is_left

# def apply_voxel_flips(ijk: np.ndarray, vol_shape, flips: tuple[bool,bool,bool]) -> np.ndarray:
#     dims = np.array(vol_shape, dtype=float)
#     out = ijk.copy()
#     for ax, do_flip in enumerate(flips):
#         if do_flip:
#             out[:, ax] = (dims[ax] - 1.0) - out[:, ax]
#     return out

# def hemi_accuracy(pred_is_left: np.ndarray, hemi_expected: np.ndarray) -> float:
#     """
#     hemi_expected entries are 'L','R', or None
#     pred_is_left is boolean array from nearest pial
#     """
#     mask = np.array([h in ("L","R") for h in hemi_expected], dtype=bool)
#     if mask.sum() == 0:
#         return np.nan
#     exp_is_left = np.array([h == "L" for h in hemi_expected[mask]], dtype=bool)
#     pred = pred_is_left[mask]
#     return float(np.mean(pred == exp_is_left))

# def choose_best_perm_and_flips(
#     pts_ijk_raw: np.ndarray,
#     vox2ras_tkr: np.ndarray,
#     lh_v: np.ndarray,
#     rh_v: np.ndarray,
#     vol_shape,
#     hemi_expected: np.ndarray,
#     lambda_hemi: float,
# ):
#     """
#     Search over:
#       - 6 perms of columns
#       - 8 voxel flips (i, j, k)
#     Score = median_dist_to_pial + lambda_hemi*(1 - hemi_acc)
#       If hemi_expected missing => score = median_dist only.

#     Returns sorted list of candidate dicts (best first).
#     """
#     dims = np.array(vol_shape, dtype=float)

#     candidates = []
#     perms = list(itertools.permutations([0,1,2], 3))
#     flips_list = list(itertools.product([False, True], repeat=3))

#     for perm in perms:
#         base = pts_ijk_raw[:, perm]

#         # early reject if totally out-of-bounds (with slack)
#         if not ((base >= -5).all() and (base <= (dims + 5)).all()):
#             continue

#         for flips in flips_list:
#             ijk = apply_voxel_flips(base, vol_shape, flips)

#             # bound check again (allow slack)
#             if not ((ijk >= -5).all() and (ijk <= (dims + 5)).all()):
#                 continue

#             tkr = voxel_to_tkr(ijk, vox2ras_tkr)
#             dist_mm, pred_is_left = nearest_pial_metrics(tkr, lh_v, rh_v)

#             med = float(np.median(dist_mm))
#             acc = hemi_accuracy(pred_is_left, hemi_expected)

#             if np.isnan(acc):
#                 score = med
#             else:
#                 score = med + lambda_hemi * (1.0 - acc)

#             candidates.append({
#                 "perm": perm,
#                 "flips": flips,
#                 "median_dist_mm": med,
#                 "hemi_acc": acc,
#                 "score": float(score),
#                 "tkr": tkr,
#                 "dist": dist_mm,
#                 "pred_is_left": pred_is_left,
#             })

#     if not candidates:
#         raise RuntimeError("No valid candidates found (all out-of-bounds).")

#     candidates.sort(key=lambda d: d["score"])
#     return candidates


# # -------------------------
# # Rendering helpers
# # -------------------------
# def make_mesh(v, f):
#     return pv.PolyData(v, np.c_[np.full(len(f), 3), f].astype(np.int64))

# def set_view(pl, view):
#     view = view.lower()
#     if view == "left":
#         pl.view_yz(negative=True)
#     elif view == "frontal":
#         pl.view_xz(negative=False)
#     elif view == "right":
#         pl.view_yz(negative=False)
#     else:
#         raise ValueError(view)
#     pl.camera.zoom(1.15)

# def render_view(lh_mesh, rh_mesh, points_tkr, view):
#     pl = pv.Plotter(off_screen=True, window_size=WINDOW_SIZE)
#     pl.set_background("white")
#     pl.add_mesh(lh_mesh, color=BRAIN_COLOR, opacity=BRAIN_OPACITY, smooth_shading=True)
#     pl.add_mesh(rh_mesh, color=BRAIN_COLOR, opacity=BRAIN_OPACITY, smooth_shading=True)
#     pl.add_points(points_tkr, color=POINT_COLOR, render_points_as_spheres=True,
#                   point_size=POINT_SIZE, opacity=POINT_OPACITY)
#     set_view(pl, view)
#     img = pl.screenshot(transparent_background=TRANSPARENT_BG, return_img=True)
#     pl.close()
#     return img

# def stitch_horiz(imgs):
#     H = min(im.shape[0] for im in imgs)
#     imgs = [im[:H] for im in imgs]
#     return np.concatenate(imgs, axis=1)


# # -------------------------
# # Core per-patient routine
# # -------------------------
# def export_and_mosaic_patient(pid: str):
#     pid = str(pid)
#     subj_dir = SHARED_ROOT / pid

#     names_path = subj_dir / "elec_recon" / f"{pid}.electrodeNames"
#     vox_path   = subj_dir / "elec_recon" / f"{pid}.LEPTOVOX"
#     if not names_path.is_file():
#         raise FileNotFoundError(f"missing {names_path}")
#     if not vox_path.is_file():
#         raise FileNotFoundError(f"missing {vox_path}")

#     mgz = pick_mgz(subj_dir)
#     img = nib.load(str(mgz))
#     vox2ras_tkr = img.header.get_vox2ras_tkr()
#     vol_shape = img.shape[:3]

#     # surfaces
#     lh_v, lh_f = read_geometry(str(subj_dir / "surf" / "lh.pial"))
#     rh_v, rh_f = read_geometry(str(subj_dir / "surf" / "rh.pial"))
#     lh_mesh = make_mesh(lh_v, lh_f)
#     rh_mesh = make_mesh(rh_v, rh_f)

#     # names + hemi labels
#     lines = read_electrode_lines_drop2(names_path)
#     names_raw, names_clean, hemi = parse_name_and_hemi(lines)

#     # LEPTOVOX coords
#     pts = read_leptovox_xyz(vox_path)
#     if pts.shape[0] != len(names_raw):
#         raise RuntimeError(f"{pid}: LEPTOVOX rows ({pts.shape[0]}) != electrodeNames contacts ({len(names_raw)})")

#     # 0/1-based detection
#     mins = pts.min(axis=0)
#     maxs = pts.max(axis=0)
#     dims = np.array(vol_shape, dtype=float)
#     looks_one_based  = (mins >= 1).all() and (maxs <= dims).all()
#     looks_zero_based = (mins >= 0).all() and (maxs <  dims).all()

#     pts_ijk_raw = pts.copy()
#     index_mode = "0-based (assumed)"
#     if looks_one_based and not looks_zero_based:
#         pts_ijk_raw -= 1.0
#         index_mode = "1-based->0-based"

#     # candidate search (perm + flips) with hemi-aware scoring
#     cands = choose_best_perm_and_flips(
#         pts_ijk_raw, vox2ras_tkr, lh_v, rh_v, vol_shape, hemi,
#         lambda_hemi=LAMBDA_HEMI,
#     )

#     best = cands[0]
#     pts_tkr = best["tkr"]
#     dist_mm = best["dist"]
#     pred_is_left = best["pred_is_left"]

#     # outputs
#     out_coords = OUT_ROOT / pid / "glassbrain" / "coords"
#     out_pngdir = OUT_ROOT / pid / "glassbrain" / "png"
#     out_candir = OUT_ROOT / pid / "glassbrain" / "png_candidates"
#     out_coords.mkdir(parents=True, exist_ok=True)
#     out_pngdir.mkdir(parents=True, exist_ok=True)
#     out_candir.mkdir(parents=True, exist_ok=True)

#     out_csv = out_coords / f"{pid}_contacts_tkrRAS.csv"
#     out_png = out_pngdir / f"{pid}_mosaic_LEPTOVOX.png"

#     # Save best CSV
#     if OVERWRITE_CSV or (not out_csv.is_file()):
#         df_out = pd.DataFrame({
#             "name_raw": names_raw,     # e.g. "FPG1 D L"
#             "name": names_clean,       # e.g. "FPG1"
#             "hemi_expected": hemi,     # L/R/None
#             "x": pts_tkr[:,0], "y": pts_tkr[:,1], "z": pts_tkr[:,2],
#             "pred_isLeft": pred_is_left.astype(int),
#             "dist_to_pial_mm": np.round(dist_mm, 2),
#             "source_space": "tkrRAS",
#             "source_provenance": f"LEPTOVOX; {index_mode}; perm={best['perm']}; flips={best['flips']}; lambda_hemi={LAMBDA_HEMI}",
#             "leptovox_file": str(vox_path),
#             "mgz_used": str(mgz),
#         })
#         df_out.to_csv(out_csv, index=False)

#     # Save best mosaic
#     if OVERWRITE_PNG or (not out_png.is_file()):
#         imgs = [render_view(lh_mesh, rh_mesh, pts_tkr, v) for v in VIEWS]
#         mosaic = stitch_horiz(imgs)
#         iio.imwrite(out_png, mosaic)

#     # Save TOP-K candidate mosaics for visual inspection
#     summary_rows = []
#     for rank, cand in enumerate(cands[:K_CANDIDATES_TO_SAVE], start=1):
#         perm = cand["perm"]
#         flips = cand["flips"]
#         med = cand["median_dist_mm"]
#         acc = cand["hemi_acc"]
#         score = cand["score"]

#         tag = f"cand{rank:02d}_perm{perm}_flip{tuple(int(b) for b in flips)}_med{med:.2f}_acc{(acc if not np.isnan(acc) else -1):.2f}_score{score:.2f}"
#         png_path = out_candir / f"{pid}_{tag}.png"

#         if OVERWRITE_CANDIDATES or (not png_path.is_file()):
#             imgs = [render_view(lh_mesh, rh_mesh, cand["tkr"], v) for v in VIEWS]
#             mosaic = stitch_horiz(imgs)
#             iio.imwrite(png_path, mosaic)

#         summary_rows.append({
#             "rank": rank,
#             "perm": perm,
#             "flips": flips,
#             "median_dist_mm": med,
#             "hemi_acc": acc,
#             "score": score,
#             "png": str(png_path),
#         })

#     df_sum = pd.DataFrame(summary_rows)
#     df_sum.to_csv(out_candir / f"{pid}_candidates_summary.csv", index=False)

#     return {
#         "pid": pid,
#         "status": "OK",
#         "n_contacts": int(len(names_raw)),
#         "index_mode": index_mode,
#         "best_perm": best["perm"],
#         "best_flips": best["flips"],
#         "median_dist_mm": float(np.median(dist_mm)),
#         "hemi_acc": best["hemi_acc"],
#         "csv": str(out_csv),
#         "png": str(out_png),
#         "candidates_dir": str(out_candir),
#     }


# # -------------------------
# # Batch run all PAT_*
# # -------------------------
# patient_ids = sorted([p.name for p in SHARED_ROOT.iterdir() if p.is_dir() and p.name.startswith("PAT_")])

# rows = []
# for pid in patient_ids:
#     try:
#         rows.append(export_and_mosaic_patient(pid))
#     except Exception as e:
#         rows.append({"pid": pid, "status": "ERROR", "error": str(e)})

# df_status = pd.DataFrame(rows)
# df_status





In [15]:
# df_status.png[0]
